# 回帰分析（演習）消費関数の推定

人は所得が増えると消費を増やす。では、所得が1万円増えたとき、消費はどれだけ増えるのだろうか。この「増えた所得のうち消費に回る割合」を限界消費性向という。

この回では、所得と消費の関係を直線で表した消費関数を、実際のデータに回帰直線をあてはめて推定する。

## 0. この課題の進め方

第12回は課題形式である。次の手順で進めること。

1. このノートブックを上から順にすべてのセルを実行する（メニューの 実行 → すべてのセルを実行、または各セルで Ctrl+Enter）。
2. 最後の「練習問題」に自分の言葉で解答し、moodle上の第12回出席小テストに入力する。

提出期限は7月7日(火)23:59とする。

## 1. 今回の問い

1. 所得（実収入）が高い都道府県ほど、消費支出は多いのか。
2. 実収入が1万円増えると、消費支出はおよそいくら増えるのか（限界消費性向）。
3. その関係は、データにどれくらいよくあてはまっているのか。

## 2. 消費関数と限界消費性向

消費 $C$ が所得 $Y$ によって決まると考え、両者の関係を直線で表したものを消費関数という。

$$
C = \alpha + \beta Y
$$

- $\beta$（傾き）を限界消費性向（MPC）という。所得が1単位増えたときに消費が増える分であり、「増えた所得のうち消費に回る割合」を表す。ふつう $0 < \beta < 1$ となる。所得が増えれば消費も増えるので $\beta$ は正になり、増えた所得のすべてが消費に回るわけではなく、一部は貯蓄されるので $\beta$ は1より小さくなる。
- $\alpha$（切片）を基礎消費という。所得が0でも生きるために必要な消費を表す。


$\alpha$ と $\beta$ は未知であり、マクロ経済学では外生的に与えられた。これを実際のデータから推定するのが回帰分析である。

## 3. ライブラリの読み込み

In [ ]:
%pip install japanize-matplotlib-jlite -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import japanize_matplotlib_jlite

## 4. 使用するデータ

総務省「家計調査年報」より、都道府県庁所在市別の勤労者世帯（世帯主が会社などに勤めている世帯）の所得と消費を用いる。

各都道府県について、1世帯当たり1か月間の実収入と消費支出（いずれも円）が公表されている。実収入は税金や社会保険料を引く前の収入である。

出典：総務省「家計調査年報」（平成13年）。都道府県庁所在市別、二人以上の世帯のうち勤労者世帯、1世帯当たり1か月間の実収入・消費支出。（社会・人口統計体系 都道府県データ）

In [ ]:
# 総務省「家計調査年報」（平成13年）都道府県庁所在市別・勤労者世帯（円/月）
# income      ：1世帯当たり1か月間の実収入（円）
# consumption ：1世帯当たり1か月間の消費支出（円）

prefecture  = ['北海道', '青森', '岩手', '宮城', '秋田', '山形', '福島', '茨城', '栃木', '群馬', '埼玉', '千葉', '東京', '神奈川', '新潟', '富山', '石川', '福井', '山梨', '長野', '岐阜', '静岡', '愛知', '三重', '滋賀', '京都', '大阪', '兵庫', '奈良', '和歌山', '鳥取', '島根', '岡山', '広島', '山口', '徳島', '香川', '愛媛', '高知', '福岡', '佐賀', '長崎', '熊本', '大分', '宮崎', '鹿児島', '沖縄']
income      = [549652, 497886, 497879, 493804, 620660, 575738, 651256, 590411, 529565, 395032, 579944, 513732, 576114, 587490, 593017, 718949, 644472, 589244, 605783, 550678, 553711, 570804, 549293, 613408, 514265, 553325, 498215, 471591, 567612, 523439, 515237, 541443, 521589, 600005, 588069, 571673, 623654, 509141, 615224, 506806, 599046, 423789, 563546, 650772, 564104, 525686, 442123]
consumption = [342975, 319097, 309422, 308764, 306995, 352590, 374364, 347800, 329915, 313539, 373238, 343458, 362268, 349458, 325461, 414457, 386077, 314249, 363210, 339957, 335511, 342492, 324827, 324456, 357206, 331812, 318782, 327449, 384230, 302720, 310044, 314231, 328542, 355255, 333308, 340550, 328308, 307132, 338271, 354822, 350943, 291320, 335763, 352291, 310045, 327691, 249810]

df = pd.DataFrame({
    "prefecture": prefecture,
    "income": income,
    "consumption": consumption,
})

df.head()

In [ ]:
# 行数と列数を確認する（47都道府県 × 3列）
print(df.shape)

## 5. 散布図で関係を見る

回帰分析の前に、まず散布図でデータの形を見る。横軸に1か月の実収入、縦軸に1か月の消費支出をとる。点は47都道府県である。

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["income"], df["consumption"], alpha=0.7)

plt.title("都道府県別の実収入と消費支出（勤労者世帯）")
plt.xlabel("1か月の実収入（円）")
plt.ylabel("1か月の消費支出（円）")
plt.grid(True)
plt.show()

## 6. 回帰分析で消費関数を推定する

消費関数 $C = \alpha + \beta Y$ を、最小二乗法で推定する。被説明変数 $Y$ を消費支出、説明変数 $X$ を実収入とする。

In [ ]:
# 被説明変数（消費支出）
Y = df["consumption"]

# 説明変数（実収入）
X = df[["income"]]

# 切片に対応する定数項を追加する
X = sm.add_constant(X)

model = sm.OLS(Y, X)
result = model.fit()

result.params

In [ ]:
coef_table = pd.DataFrame({
    "推定値": result.params,
})
coef_table

推定された2つの値のうち、

- const が基礎消費 $\alpha$（実収入が0のときの消費、円）
- income が限界消費性向 $\beta$（実収入が1円増えたときに消費が増える分）

である。$\beta$ は「実収入が1円増えると消費が $\beta$ 円増える」と読む。10000倍すれば「実収入が1万円増えると消費が何円増えるか」になる。

## 7. 回帰直線を散布図に重ねる

In [ ]:
x_line = np.linspace(df["income"].min(), df["income"].max(), 100)
y_line = result.params["const"] + result.params["income"] * x_line

plt.figure(figsize=(8, 6))
plt.scatter(df["income"], df["consumption"], alpha=0.7, label="実際のデータ")
plt.plot(x_line, y_line, color="tab:red", label="推定された消費関数")

plt.title("実収入と消費支出、推定された消費関数")
plt.xlabel("1か月の実収入（円）")
plt.ylabel("1か月の消費支出（円）")
plt.legend()
plt.grid(True)
plt.show()

## 8. あてはまりの良さ（決定係数）

決定係数 $R^2$ は、消費のばらつきのうち、実収入によって説明できる割合を表す。0から1のあいだの値をとり、1に近いほど直線がデータによくあてはまっている。

In [ ]:
print("決定係数 R^2:", round(result.rsquared, 4))

## 9. 限界消費性向は統計的に意味があるか（検定）

推定された $\beta$ が、「本当は実収入と消費に関係がない（$\beta = 0$）のに、たまたまこの値が出ただけ」ではないことを確かめる。第10回で学んだ $t$ 検定を使う。今回は47都道府県のデータ（標本の数は47、自由度は45）である。

帰無仮説は「$\beta = 0$（実収入は消費に影響しない）」である。$t$ 値が大きく、$p$ 値が小さいほど、この帰無仮説は否定され、実収入と消費の関係は意味があることが示唆される。

In [ ]:
test_table = pd.DataFrame({
    "推定値": result.params,
    "標準誤差": result.bse,
    "t値": result.tvalues,
    "p値": result.pvalues,
})
test_table

In [ ]:
# 傾き（限界消費性向）について5%有意水準で判定する
alpha = 0.05
p_beta = result.pvalues["income"]

if p_beta < alpha:
    print("p値は", round(p_beta, 6), "で、5%有意水準で帰無仮説を棄却する（関係は有意である）")
else:
    print("p値は", round(p_beta, 6), "で、5%有意水準で帰無仮説を棄却しない")

In [ ]:
# 限界消費性向の95%信頼区間
conf = result.conf_int(alpha=0.05)
conf.columns = ["95%信頼区間 下限", "95%信頼区間 上限"]
conf

## 練習問題

次の問いに、これまでの出力を見ながら自分の言葉で答えなさい。

1. この分析で推定された限界消費性向はおよそいくつか。その数値の意味を「実収入」「消費」という言葉を使って説明しなさい。
2. この分析によると、実収入が1万円（10000円）増えると、消費支出はおよそ（　　　）円増える。
3. 限界消費性向が $0 < \beta < 1$ におさまっているのはなぜか。増えた所得は消費以外に何に回ると考えられるか。